# Side-View Squat Random Forest

This notebook is the readable, step-by-step companion to `train_squat_quality_model.py`. It trains one binary Random Forest that predicts whether a completed squat repetition is **GOOD (`1`)** or **BAD (`0`)**. Hard-coded rules handle the specific feedback reason.

The notebook imports the Python trainer instead of copying its implementation, so the command-line and notebook workflows cannot drift apart.

## 1. Imports and paths

The path setup works whether Jupyter starts in the repository root or in this exercise folder.

In [1]:
from pathlib import Path
import sys

import pandas as pd

working_directory = Path.cwd().resolve()
exercise_directory = (
    working_directory / 'exercises' / 'side_view_squat'
    if (working_directory / 'exercises' / 'side_view_squat').is_dir()
    else working_directory
)
repository_root = exercise_directory.parent.parent
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

from exercises.side_view_squat.config import FEATURE_COLUMNS, TARGET_COLUMN
from exercises.side_view_squat.train_squat_quality_model import (
    build_model,
    cv_strategy,
    load_dataset,
    recording_groups,
    train,
)

dataset_path = exercise_directory / 'data' / 'side_squat_reps.csv'
model_path = exercise_directory / 'models' / 'side_squat_hybrid_quality_v1.pkl'

print(f'Dataset: {dataset_path}')
print(f'Model output: {model_path}')

Dataset: C:\Users\user\OneDrive\Desktop\form trial\exercises\side_view_squat\data\side_squat_reps.csv
Model output: C:\Users\user\OneDrive\Desktop\form trial\exercises\side_view_squat\models\side_squat_hybrid_quality_v1.pkl


## 2. Load and inspect the data

Each row represents one completed squat repetition. The loader removes rows missing required values and checks that `is_good` contains only `0` and `1`.

In [2]:
squat_data = load_dataset(dataset_path)
print(f'Usable repetitions: {len(squat_data)}')
print('Label counts (0 = BAD, 1 = GOOD):')
display(squat_data[TARGET_COLUMN].value_counts().sort_index().rename('count').to_frame())
display(squat_data[FEATURE_COLUMNS + [TARGET_COLUMN]].head())

Usable repetitions: 95
Label counts (0 = BAD, 1 = GOOD):


,count
is_good,
0,47
1,48


,min_knee_angle,max_knee_angle,knee_rom,knee_angle_at_bottom,max_descent_knee_velocity,max_ascent_knee_velocity,min_hip_angle,max_hip_angle,hip_rom,hip_angle_at_bottom,...,rep_duration,descent_duration,bottom_duration,ascent_duration,descent_ascent_ratio,pose_visibility_mean,pose_visibility_min,valid_frame_ratio,valid_frame_count,is_good
0,91.120977,170.182283,79.061306,91.255128,169.775126,173.779109,86.968250,165.486234,78.517985,86.968250,...,1.906,0.609,0.203,0.937,0.649947,0.989368,0.985561,0.909091,20.0,1
1,116.362934,171.989278,55.626343,118.285849,0.000000,199.604397,110.676030,164.134820,53.458790,115.256970,...,1.594,0.000,0.563,0.844,0.000000,0.988121,0.982859,1.000000,19.0,0
2,89.176159,172.903918,83.727760,89.176159,45.638135,181.431264,84.124992,167.622864,83.497872,84.124992,...,1.515,0.359,0.078,0.890,0.403371,0.988136,0.982639,0.944444,17.0,1
3,93.404538,166.163350,72.758812,93.404538,130.863802,295.310922,89.421210,162.952356,73.531146,89.421210,...,1.593,0.328,0.187,0.890,0.368539,0.991162,0.986463,1.000000,19.0,0
4,104.230793,167.769198,63.538405,104.230793,157.969353,187.860382,97.962297,165.530247,67.567949,97.962297,...,1.719,0.266,0.187,0.797,0.333752,0.990814,0.987083,0.850000,17.0,1


## 3. Review features and validation

Random Forests do not require feature scaling. Validation keeps repetitions from the same participant/session/recording together when enough groups exist; otherwise it falls back to stratified repetition-level folds.

In [3]:
display(pd.DataFrame({'feature': FEATURE_COLUMNS}))

groups = recording_groups(squat_data)
cross_validator, cv_groups, validation_method = cv_strategy(
    squat_data[TARGET_COLUMN], groups
)
print(f'Validation method: {validation_method}')
print(f'Number of folds: {cross_validator.get_n_splits()}')
print(f'Recording groups: {len(set(groups)) if groups is not None else 0}')

,feature
0,min_knee_angle
1,max_knee_angle
2,knee_rom
3,knee_angle_at_bottom
4,max_descent_knee_velocity
5,max_ascent_knee_velocity
6,min_hip_angle
7,max_hip_angle
8,hip_rom
9,hip_angle_at_bottom


Validation method: stratified_group
Number of folds: 3
Recording groups: 3


## 4. Review the Random Forest configuration

In [4]:
random_forest = build_model()
random_forest

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",400
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",8
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y_

## 5. Evaluate, train, and save

This runs out-of-fold evaluation, fits the final model on all usable repetitions, and saves the complete model bundle.

In [5]:
model_bundle = train(dataset_path, model_path)
model_bundle['evaluation_metrics']

Out-of-fold evaluation (not training accuracy):
  accuracy=0.853 precision=0.827 recall=0.896 f1=0.860
  confusion_matrix [rows actual BAD/GOOD, cols predicted BAD/GOOD]=[[38, 9], [5, 43]]
Saved model bundle: C:\Users\user\OneDrive\Desktop\form trial\exercises\side_view_squat\models\side_squat_hybrid_quality_v1.pkl


{'accuracy': 0.8526315789473684,
 'precision': 0.8269230769230769,
 'recall': 0.8958333333333334,
 'f1': 0.86,
 'confusion_matrix': [[38, 9], [5, 43]],
 'predicted_good_probabilities': [0.955028,
  0.07488,
  0.98083,
  0.817814,
  0.566089,
  0.720561,
  0.063474,
  0.72657,
  0.712104,
  0.15092,
  0.594769,
  0.159713,
  0.153211,
  0.248972,
  0.232798,
  0.679487,
  0.688178,
  0.700474,
  0.766083,
  0.651388,
  0.668109,
  0.752248,
  0.203418,
  0.291375,
  0.126991,
  0.561612,
  0.768706,
  0.700102,
  0.217829,
  0.231104,
  0.197469,
  0.694347,
  0.681262,
  0.660267,
  0.331851,
  0.21391,
  0.076781,
  0.762352,
  0.674907,
  0.195854,
  0.143427,
  0.741461,
  0.208738,
  0.235616,
  0.233301,
  0.179227,
  0.196411,
  0.185919,
  0.184497,
  0.714623,
  0.769458,
  0.643338,
  0.692327,
  0.678691,
  0.535549,
  0.546677,
  0.063812,
  0.320647,
  0.578267,
  0.464378,
  0.722635,
  0.281079,
  0.55912,
  0.402348,
  0.651958,
  0.533597,
  0.688453,
  0.787515,
  0.03

## 6. Inspect feature importance

Feature importance shows which inputs the fitted forest used most. It describes this model and dataset; it does not prove that a feature causes good or bad form.

In [6]:
feature_importance = (
    pd.DataFrame({
        'feature': model_bundle['feature_columns'],
        'importance': model_bundle['quality_model'].feature_importances_,
    })
    .sort_values('importance', ascending=False, ignore_index=True)
)
display(feature_importance)

,feature,importance
0,hip_knee_depth_at_bottom,0.139105
1,hip_angle_at_bottom,0.110825
2,min_hip_angle,0.072542
3,torso_lean_at_bottom,0.067217
4,knee_angle_at_bottom,0.057600
5,max_torso_lean,0.056822
6,hip_rom,0.055000
7,heel_lift_max,0.054727
8,knee_rom,0.049463
9,heel_lift_bottom_fraction,0.041930
